In [4]:
import os
from langchain_openrouter import ChatOpenRouter

os.environ["OPENROUTER_API_KEY"] = "sk-or-w2-dfkkgfyujkiu-yfykjhlli"

model = ChatOpenRouter(model="openai/gpt-4o-mini", # "auto" can be an option
                       max_tokens=200,
                       temperature=0.7,)


## Tools

Tools extend what agents can do—letting them fetch real-time data, execute code, query external databases, and take actions in the world.

Under the hood, tools are callable functions with well-defined inputs and outputs that get passed to a chat model. The model decides when to invoke a tool based on the conversation context, and what input arguments to provide.


### Create tools
#### Basic tool definition
The simplest way to create a tool is with the @tool decorator. By default, the function’s docstring becomes the tool’s description that helps the model understand when to use it:

In [1]:
from langchain.tools import tool

@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

### Customize tool properties
#### Custom tool name

By default, the tool name comes from the function name. Override it when you need something more descriptive:
#### Custom tool description
Override the auto-generated tool description for clearer model guidance:

In [2]:
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

In [3]:
weather_schema = {
    "type": "object",
    "properties": {
        "location": {"type": "string"},
        "units": {"type": "string"},
        "include_forecast": {"type": "boolean"}
    },
    "required": ["location", "units", "include_forecast"]
}

@tool(args_schema=weather_schema)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [20]:
from langchain.messages import AIMessage
from langchain.messages import HumanMessage
from langchain.messages import ToolMessage

model_with_tool = model.bind_tools([get_weather, search_database, calc])
messages = [HumanMessage("What's the weather in New York for the next 5 days? Measure the temperature in celsius unit")]

response = model_with_tool.invoke(messages)
messages.append(AIMessage(content=response.content, tool_calls=response.tool_calls))
observation = None
for tool in response.tool_calls:
    print(f"Tool call: {tool}")
    observation = get_weather.invoke(tool)

print("Observation from tool call:", observation)
messages.append(ToolMessage(content=observation, tool_call_id=response.tool_calls[0]["id"]))

response = model_with_tool.invoke(messages)
print("Final response from model:", response.content)

Tool call: {'name': 'get_weather', 'args': {'location': 'New York', 'units': 'metric', 'include_forecast': True}, 'id': 'call_ub8eTIEFT3DkGn4nxdzaTPFU', 'type': 'tool_call'}
Observation from tool call: content='Current weather in New York: 72 degrees M\nNext 5 days: Sunny' name='get_weather' tool_call_id='call_ub8eTIEFT3DkGn4nxdzaTPFU'
Final response from model: The current weather in New York is 22 degrees Celsius. For the next 5 days, the forecast is sunny.



### Short-term memory (State)
State represents short-term memory that exists for the duration of a conversation. It includes the message history and any custom fields you define in your graph state.


#### Access state
Tools can access the current conversation state using runtime.state:

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage

@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    messages = runtime.state["messages"]

    # Find the last human message
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content

    return "No user messages found"

# Access custom state fields
@tool
def get_user_preference(
    pref_name: str,
    runtime: ToolRuntime
) -> str:
    """Get a user preference value."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")


#### Update state
Use Command to update the agent’s state. This is useful for tools that need to update custom state fields. Include a ToolMessage in the update so the model can see the result of the tool call:

In [ ]:
from langchain.agents import AgentState
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command


class CustomState(AgentState):
    user_name: str


@tool
def set_user_name(new_name: str, runtime: ToolRuntime[None, CustomState]) -> Command:
    """Set the user's name in the conversation state."""
    return Command(
        update={
            "user_name": new_name,
            "messages": [
                ToolMessage(
                    content=f"User name set to {new_name}.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


#### Context
Context provides immutable configuration data that is passed at invocation time. Use it for user IDs, session details, or application-specific settings that shouldn’t change during a conversation.
Access context through runtime.context:

In [21]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime


USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com"
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com"
    }
}

@dataclass
class UserContext:
    user_id: str

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return f"Account holder: {user['name']}\nType: {user['account_type']}\nBalance: ${user['balance']}"
    return "User not found"

agent = create_agent(
    model,
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt="You are a financial assistant."
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current balance?"}]},
    context=UserContext(user_id="user123")
)

In [24]:
print(result)
print(result.ToolMessage)

{'messages': [HumanMessage(content="What's my current balance?", additional_kwargs={}, response_metadata={}, id='d0511ba7-dc88-4f74-bab0-1b8940c219c0'), AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-4o-mini', 'id': 'gen-1778591315-T2PhDF0HdD7h2A6uMrIH', 'created': 1778591315, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'system_fingerprint': 'fp_20688f8405'}, id='lc_run--019e1c4d-b3be-71e2-8a69-ce1a94fef37b-0', tool_calls=[{'name': 'get_account_info', 'args': {}, 'id': 'call_R6Q6XEkPBPGPJPQHOC6nLIyD', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 11, 'total_tokens': 61, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}), ToolMessage(content='Account holder: Alice Johnson\nType: Premium\nBalance: $5000', name='get_account_info', id='d5877d74-2392-418e-be47-d478ee331081', too

AttributeError: 'dict' object has no attribute 'ToolMessage'